In [38]:
ID = 4

In [39]:
# inference.py
import torch
import yaml
from model import myTransformer

# -------------------------
# Load config
# -------------------------
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_cfg = config["data"]
model_cfg = config["model"]
device = config["training"]["device"]

INPUT_WINDOW = data_cfg["input_window"]
OUTPUT_WINDOW = data_cfg["output_window"]

# -------------------------
# Build model (SAME as training)
# -------------------------
model = myTransformer(
    input_dim=8,
    output_dim=1,
    input_window=12,
    output_window=6,
    d_model=64,          # 🔥 NON 128
    nhead=4,
    num_layers=2,        # 🔥 NON 4
    dropout=0.1          # il dropout non conta in eval
)


model.to(device)

# -------------------------
# Load trained weights
# -------------------------
state_dict = torch.load(f"client_{ID}_final_model.pth", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()


model.eval()
print("✅ Model loaded and ready for inference")


✅ Model loaded and ready for inference


In [6]:
import pandas as pd

# -------------------------
# Load CSV
# -------------------------
csv_path = "../../../data/events_dataset23.csv"

df = pd.read_csv(csv_path)

In [14]:
import pandas as pd

csv_path = "../../../data/events_dataset23.csv"

df = pd.read_csv(csv_path, sep=";")

# pulizia nomi colonne
df.columns = df.columns.str.strip().str.lower()

print(df.columns.tolist())
print(df.head())


['date', 'time', 'barometer', 'temperature', 'wind_speed', 'rain_rate', 'datetime', 'station_id']
         date      time  barometer  temperature  wind_speed  rain_rate  \
0  23-12-2025  00:00:00     1008.3         13.9         2.0        0.0   
1  23-12-2025  00:10:00     1008.3         14.0         1.6        0.0   
2  23-12-2025  00:20:00     1008.3         13.6         1.4        0.0   
3  23-12-2025  00:30:00     1008.2         13.4         1.1        0.0   
4  23-12-2025  00:40:00     1008.3         13.7         1.2        0.0   

              datetime  station_id  
0  23-12-2025 00:00:00           1  
1  23-12-2025 00:10:00           1  
2  23-12-2025 00:20:00           1  
3  23-12-2025 00:30:00           1  
4  23-12-2025 00:40:00           1  


In [15]:
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

df = df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
df["hour"] = df["datetime"].dt.hour


/var/folders/h_/57x0p6rs0z7crnpsbzlw5nd40000gn/T/ipykernel_32331/34628817.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["datetime"] = pd.to_datetime(df["datetime"], utc=True)


In [16]:
WINDOW = 12
station_windows = {}

for station_id, df_station in df.groupby("station_id"):
    df_station = df_station.reset_index(drop=True)

    idx_14 = df_station.index[df_station["hour"] == 14]

    if len(idx_14) == 0:
        print(f"⚠️ Station {station_id}: nessuna riga alle 14 UTC")
        continue

    idx = idx_14[0]

    if idx < WINDOW:
        print(f"⚠️ Station {station_id}: non abbastanza dati prima delle 14")
        continue

    window_df = df_station.iloc[idx-WINDOW:idx].copy()
    station_windows[station_id] = window_df

    print(f"✅ Station {station_id}: finestra OK")


✅ Station 1: finestra OK
✅ Station 2: finestra OK
✅ Station 3: finestra OK
✅ Station 4: finestra OK


In [18]:
station_windows[1]

,date,time,barometer,temperature,wind_speed,rain_rate,datetime,station_id,hour
72,23-12-2025,12:00:00,1005.1,17.2,1.8,0.0,2025-12-23 12:00:00+00:00,1,12
73,23-12-2025,12:10:00,1005.0,17.4,2.1,0.0,2025-12-23 12:10:00+00:00,1,12
74,23-12-2025,12:20:00,1005.0,17.3,2.1,0.0,2025-12-23 12:20:00+00:00,1,12
75,23-12-2025,12:30:00,1004.9,17.2,3.2,0.0,2025-12-23 12:30:00+00:00,1,12
76,23-12-2025,12:40:00,1004.8,17.1,2.3,0.0,2025-12-23 12:40:00+00:00,1,12
77,23-12-2025,12:50:00,1004.8,16.9,2.5,0.0,2025-12-23 12:50:00+00:00,1,12
78,23-12-2025,13:00:00,1004.9,15.5,3.5,0.0,2025-12-23 13:00:00+00:00,1,13
79,23-12-2025,13:10:00,1004.8,14.4,3.9,0.0,2025-12-23 13:10:00+00:00,1,13
80,23-12-2025,13:20:00,1004.9,14.1,4.3,0.0,2025-12-23 13:20:00+00:00,1,13
81,23-12-2025,13:30:00,1005.0,13.7,4.1,0.0,2025-12-23 13:30:00+00:00,1,13


In [19]:
station_windows[2]

,date,time,barometer,temperature,wind_speed,rain_rate,datetime,station_id,hour
72,23-12-2025,12:00:00,1008.9,14.2,4.7,0.2,2025-12-23 12:00:00+00:00,2,12
73,23-12-2025,12:10:00,1008.8,13.5,4.1,0.4,2025-12-23 12:10:00+00:00,2,12
74,23-12-2025,12:20:00,1008.8,13.3,3.0,1.0,2025-12-23 12:20:00+00:00,2,12
75,23-12-2025,12:30:00,1008.7,13.0,2.7,1.2,2025-12-23 12:30:00+00:00,2,12
76,23-12-2025,12:40:00,1008.4,13.1,2.3,0.6,2025-12-23 12:40:00+00:00,2,12
77,23-12-2025,12:50:00,1008.4,13.1,1.9,0.2,2025-12-23 12:50:00+00:00,2,12
78,23-12-2025,13:00:00,1008.5,13.0,1.1,0.2,2025-12-23 13:00:00+00:00,2,13
79,23-12-2025,13:10:00,1008.2,12.9,1.5,1.4,2025-12-23 13:10:00+00:00,2,13
80,23-12-2025,13:20:00,1008.2,12.8,1.5,0.4,2025-12-23 13:20:00+00:00,2,13
81,23-12-2025,13:30:00,1008.3,12.6,1.7,0.4,2025-12-23 13:30:00+00:00,2,13


In [20]:
station_windows[3]

,date,time,barometer,temperature,wind_speed,rain_rate,datetime,station_id,hour
71,23-12-2025,12:00:00,912.4,9.9,3.1,0.0,2025-12-23 12:00:00+00:00,3,12
72,23-12-2025,12:10:00,912.3,10.0,2.7,0.0,2025-12-23 12:10:00+00:00,3,12
73,23-12-2025,12:20:00,912.2,9.9,4.2,0.0,2025-12-23 12:20:00+00:00,3,12
74,23-12-2025,12:30:00,912.1,9.8,2.5,0.0,2025-12-23 12:30:00+00:00,3,12
75,23-12-2025,12:40:00,912.0,9.9,5.5,0.0,2025-12-23 12:40:00+00:00,3,12
76,23-12-2025,12:50:00,911.9,9.9,4.1,0.0,2025-12-23 12:50:00+00:00,3,12
77,23-12-2025,13:00:00,911.9,9.9,3.3,0.0,2025-12-23 13:00:00+00:00,3,13
78,23-12-2025,13:10:00,911.8,10.0,2.3,0.0,2025-12-23 13:10:00+00:00,3,13
79,23-12-2025,13:20:00,911.7,10.2,3.0,0.0,2025-12-23 13:20:00+00:00,3,13
80,23-12-2025,13:30:00,911.7,10.3,4.7,0.0,2025-12-23 13:30:00+00:00,3,13


In [21]:
station_windows[4]

,date,time,barometer,temperature,wind_speed,rain_rate,datetime,station_id,hour
72,23-12-2025,12:00:00,997.4,16.2,2.8,0.0,2025-12-23 12:00:00+00:00,4,12
73,23-12-2025,12:10:00,997.3,14.7,6.1,0.0,2025-12-23 12:10:00+00:00,4,12
74,23-12-2025,12:20:00,997.3,14.2,6.2,0.0,2025-12-23 12:20:00+00:00,4,12
75,23-12-2025,12:30:00,997.3,13.9,5.9,0.0,2025-12-23 12:30:00+00:00,4,12
76,23-12-2025,12:40:00,997.3,13.7,6.3,0.0,2025-12-23 12:40:00+00:00,4,12
77,23-12-2025,12:50:00,997.3,13.4,7.8,0.0,2025-12-23 12:50:00+00:00,4,12
78,23-12-2025,13:00:00,997.6,12.7,4.7,3.4,2025-12-23 13:00:00+00:00,4,13
79,23-12-2025,13:10:00,997.3,12.3,2.6,1.2,2025-12-23 13:10:00+00:00,4,13
80,23-12-2025,13:20:00,997.2,12.0,2.8,1.2,2025-12-23 13:20:00+00:00,4,13
81,23-12-2025,13:30:00,997.1,11.9,3.1,1.0,2025-12-23 13:30:00+00:00,4,13


In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# colonne numeriche da standardizzare
num_cols = ["barometer", "temperature", "wind_speed"]

# inizializziamo scaler (puoi caricare lo stesso se salvato col training)
scaler = StandardScaler()

processed_windows = {}  # per salvare finestre preprocessate

for station_id, window_df in station_windows.items():
    df = window_df.copy()

    # --- Standardizzazione numerica ---
    df[num_cols] = scaler.fit_transform(df[num_cols])

    # --- Trasformazione rain_rate ---
    df["rain_rate"] = np.log1p(df["rain_rate"])

    # --- Parsing datetime già presente ---
    # se non c'è, puoi usare df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    dt = df["datetime"]

    # --- Feature temporali ---
    df["hour"] = dt.dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["month"] = dt.dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    # --- Selezione colonne finali per il modello ---
    feature_cols = [
        "barometer", "temperature", "wind_speed",
        "rain_rate", "hour_sin", "hour_cos",
        "month_sin", "month_cos"
    ]
    df_model = df[feature_cols].copy()

    processed_windows[station_id] = df_model

    print(f"✅ Station {station_id}: finestra preprocessata ({df_model.shape})")


✅ Station 1: finestra preprocessata ((12, 8))
✅ Station 2: finestra preprocessata ((12, 8))
✅ Station 3: finestra preprocessata ((12, 8))
✅ Station 4: finestra preprocessata ((12, 8))


In [23]:
processed_windows[1]

,barometer,temperature,wind_speed,rain_rate,hour_sin,hour_cos,month_sin,month_cos
72,1.341641,0.959579,-1.384037,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
73,0.447214,1.083396,-1.028142,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
74,0.447214,1.021487,-1.028142,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
75,-0.447214,0.959579,0.276807,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
76,-1.341641,0.897671,-0.790878,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
77,-1.341641,0.773854,-0.553615,0.0,1.224647e-16,-1.000000,-2.449294e-16,1.0
78,-0.447214,-0.092862,0.632703,0.0,-2.588190e-01,-0.965926,-2.449294e-16,1.0
79,-1.341641,-0.773854,1.107230,0.0,-2.588190e-01,-0.965926,-2.449294e-16,1.0
80,-0.447214,-0.959579,1.581757,0.0,-2.588190e-01,-0.965926,-2.449294e-16,1.0
81,0.447214,-1.207212,1.344493,0.0,-2.588190e-01,-0.965926,-2.449294e-16,1.0


In [40]:
import torch

# --- Prendiamo la finestra preprocessata per station 1 ---
df_model = processed_windows[ID]  # shape [12, 8]

# --- Convertiamo in tensor [1, 12, 8] ---
X = torch.tensor(df_model.values, dtype=torch.float32).unsqueeze(0)  # batch dimension
print(f"Input tensor shape: {X.shape}")  # dovrebbe essere [1, 12, 8]

# --- Inferenza con modello già caricato ---
model.eval()  # assicurati che sia in eval
with torch.no_grad():
    prediction = model(X)

print(f"Output tensor shape: {prediction.shape}")  # [1, 6, 1]
print("Predicted rain_rate (log1p scale):")
print(prediction)


Input tensor shape: torch.Size([1, 12, 8])
Output tensor shape: torch.Size([1, 6, 1])
Predicted rain_rate (log1p scale):
tensor([[[0.5505],
         [0.3307],
         [0.2925],
         [0.2371],
         [0.2369],
         [0.2225]]])


In [41]:
import pandas as pd
import torch

# --- Parametri ---
station_id = ID
longitude = 14.1691129  # sostituire con coordinate reali
latitude = 40.8171821   # sostituire con coordinate reali
output_csv = "23-12-2025-rain-rate-predictions.csv"

# --- Prediction già calcolata (processed_windows[1]) ---
pred_log = prediction[0, :, 0]        # [6] log1p scale
pred_original = torch.expm1(pred_log) # scala originale

# --- Generiamo manualmente i timestamp ---
timestamps = pd.date_range(
    start="2025-12-23 14:00:00",
    periods=6,
    freq="10min",
    tz="UTC"
)

# --- Creiamo DataFrame da salvare ---
df_out = pd.DataFrame({
    "WEATHER_STATION_ID": [station_id]*6,
    "ISO_8601_TIMESTAMP": timestamps.strftime("%Y-%m-%dT%H:%M:%S%z"),
    "LONGITUDE": [longitude]*6,
    "LATITUDE": [latitude]*6,
    "RAINRATE_VALUE": pred_original.numpy()
})

# --- Salvataggio CSV in modalità append ---
try:
    # se il file non esiste, lo crea con header
    with open(output_csv, 'x') as f:
        df_out.to_csv(f, index=False)
except FileExistsError:
    # se esiste, appende senza header
    df_out.to_csv(output_csv, mode='a', header=False, index=False)

print(f"✅ Predictions salvate in '{output_csv}'")
print(df_out)


✅ Predictions salvate in '23-12-2025-rain-rate-predictions.csv'
   WEATHER_STATION_ID        ISO_8601_TIMESTAMP  LONGITUDE   LATITUDE  \
0                   4  2025-12-23T14:00:00+0000  14.169113  40.817182   
1                   4  2025-12-23T14:10:00+0000  14.169113  40.817182   
2                   4  2025-12-23T14:20:00+0000  14.169113  40.817182   
3                   4  2025-12-23T14:30:00+0000  14.169113  40.817182   
4                   4  2025-12-23T14:40:00+0000  14.169113  40.817182   
5                   4  2025-12-23T14:50:00+0000  14.169113  40.817182   

   RAINRATE_VALUE  
0        0.734108  
1        0.391896  
2        0.339772  
3        0.267607  
4        0.267283  
5        0.249207  
